In [2]:
import pandas as pd
import numpy as np
import psycopg2
import sqlalchemy as db
from sqlalchemy import create_engine
import yaml

In [3]:
with open('C:\\Users\\dylan\\OneDrive\\Documentos\\mensajeria\\config.yml', 'r') as f:
    config = yaml.safe_load(f)
    config_co = config['mensajeria']
    config_etl = config['ETL_PRO']

# Construct the database URL
url_co = (f"{config_co['drivername']}://{config_co['user']}:{config_co['password']}@{config_co['host']}:"
          f"{config_co['port']}/{config_co['dbname']}")
url_etl = (f"{config_etl['drivername']}://{config_etl['user']}:{config_etl['password']}@{config_etl['host']}:"
           f"{config_etl['port']}/{config_etl['dbname']}")
# Create the SQLAlchemy Engine
mensajeria = create_engine(url_co)
etl_conn = create_engine(url_etl)

In [ ]:
def generar_dim_hora() -> pd.DataFrame:
    
    minutos = pd.date_range(start='00:00', end='23:59', freq='1min')
    
    df = pd.DataFrame({'tiempo': minutos})
    
    # Clave sustituta: HHMM como entero (0 a 2359)
    df['key_minutos'] = df['tiempo'].dt.strftime('%H%M').astype(int)
    
    # Atributos
    df['hora']         = df['tiempo'].dt.hour
    df['minutos']      = df['tiempo'].dt.minute
    df['hora_formato'] = df['tiempo'].dt.strftime('%H:%M')   # '14:30'
    
    # Franjas horarias — útil para análisis de mensajería
    df['franja'] = pd.cut(
        df['hora'],
        bins=      [0,  6, 12, 14, 18, 21, 24],
        labels=    ['madrugada', 'mañana', 'mediodía', 'tarde', 'noche', 'noche alta'],
        right=False
    )
    
    df['es_hora_habil'] = (df['hora'] >= 8) & (df['hora'] < 18)
    
    return df[['key_minutos', 'hora', 'minutos',
               'hora_formato', 'franja', 'es_hora_habil']]


dim_hora = generar_dim_hora()

print(dim_hora.shape)   # (1440, 6) → siempre exactamente 1440 filas
print(dim_hora.tail())

(1440, 6)
   key_minutos  hora  minutos hora_formato     franja  es_hora_habil
0            0     0        0        00:00  madrugada          False
1            1     0        1        00:01  madrugada          False
2            2     0        2        00:02  madrugada          False
3            3     0        3        00:03  madrugada          False
4            4     0        4        00:04  madrugada          False


In [6]:
dim_hora.tail()

,key_minutos,hora,minutos,hora_formato,franja,es_hora_habil
1435,2355,23,55,23:55,noche alta,False
1436,2356,23,56,23:56,noche alta,False
1437,2357,23,57,23:57,noche alta,False
1438,2358,23,58,23:58,noche alta,False
1439,2359,23,59,23:59,noche alta,False
